# 02 · Ticker extraction

**Goal:** show how the conservative extractor turns free text into ticker mentions, with an auditable confidence breakdown. The hard cases: `$cashtags` vs bare words, English-word tickers (`BE`, `AI`, `ON`, `IT`), blocklisted acronyms (`DD`, `CEO`), and company-name aliases (`Bloom Energy`→BE, `Nebius`→NBIS).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
import pandas as pd, numpy as np
from reddit_hype.config import load_settings, credential_report
from reddit_hype.utils import read_parquet_or_empty
s = load_settings()
print(credential_report())
from reddit_hype.ticker_extractor import TickerExtractor
uni = read_parquet_or_empty(s.path('ticker_universe'))
ex = TickerExtractor.from_settings(s, uni if not uni.empty else None)

### Worked examples — note which clear the confidence threshold and why

In [ ]:
examples = [
  '$NVDA calls printing, I bought shares',
  'I bought NVDA shares today, holding long',
  'Just BE patient and hold',            # BE = English word -> rejected
  '$BE hydrogen play looks strong',       # cashtag rescues it
  'great DD on this name',                # DD blocklisted
  'Bloom Energy looks cheap, buying calls',
  'Nebius is the GPU cloud play',
]
for t in examples:
    ms = ex.extract(t)
    print(repr(t))
    for m in ms:
        print('   ->', m.ticker, round(m.confidence,3), m.method, m.components)
    if not ms: print('   -> (no tickers)')

### Mention table summary

In [ ]:
m = read_parquet_or_empty(s.path('mentions'))
print('mentions:', len(m), '| tickers:', m['ticker'].nunique() if not m.empty else 0)
m['ticker'].value_counts().head(15) if not m.empty else 'run make extract-mentions'